# Figure 5 notebook

This notebook consolidates Figure 5 from `08b_MI_subsampling.ipynb` (panels `a-d`) and `08c_3Layer_Network.ipynb` (panel `e`). It recreates the panel images in one place and assembles a clean composite mock matching the attached reference layout.

## What this notebook does

1. Loads the cached Figure 5 analysis outputs already saved in `Notebooks/`.
2. Recreates panels `a` through `e` without rerunning the expensive subsampling or alphabet-discovery analyses.
3. Saves each panel into `Figures/` as `Figure5_panel_*.png`.
4. Assembles a final `Figure5_mock.png` preview with the same panel arrangement as the reference figure.

This notebook assumes the project is run from either the repository root or the `Notebooks/` directory.

In [ ]:
import pickle
import sys
from pathlib import Path

import distinctipy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "Data"
FIG_DIR = ROOT / "Figures"
NOTEBOOK_DIR = ROOT / "Notebooks"
FIG_DIR.mkdir(parents=True, exist_ok=True)

for repo_path in [ROOT / "Repos" / "TMG", ROOT / "Repos" / "max_info_atlas" / "src"]:
    repo_str = str(repo_path)
    if repo_str not in sys.path:
        sys.path.insert(0, repo_str)

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.grid": False,
        "savefig.facecolor": "white",
        "font.size": 10,
    }
)

from TMG.Analysis.TissueGraph import TissueMultiGraph
from TMG.Utils.coloru import rgb_array_to_hex
from TMG.Utils.networkvizu import plot_three_layers
from TMG.Utils.tmgu import correct_mi_bias, list_entropy, summarize_saturation_threshold

PANEL_PATHS = {label: FIG_DIR / f"Figure5_panel_{label}.png" for label in "abcde"}
PANEL_PATHS["mock"] = FIG_DIR / "Figure5_mock.png"

CACHE_PATHS = {
    "mi_c": DATA_DIR / "mi_c.npy",
    "mi_r": DATA_DIR / "mi_r.npy",
    "alphabet_results": DATA_DIR / "alphabet_results.pkl",
    "region_alphabet_results": DATA_DIR / "region_alphabet_results.pkl",
    "matrix_gene_cell": DATA_DIR / "matrix_gene_cell_mi_based_mapping.csv",
    "matrix_cell_region": DATA_DIR / "matrix_cell_region_mi_based_mapping.csv",
}

ALPHABET_THRESHOLD_BITS = 0.90


def require_path(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required cached file: {path}")
    return path


def load_pickle(path):
    with open(path, "rb") as handle:
        return pickle.load(handle)


def save_png(fig, path):
    fig.savefig(path, dpi=300, bbox_inches="tight")
    return path


def show_image(ax, image_path):
    image = plt.imread(image_path)
    ax.imshow(image)
    ax.set_axis_off()


def add_panel_label(ax, label):
    ax.text(
        -0.06,
        1.03,
        label,
        transform=ax.transAxes,
        fontsize=16,
        fontweight="bold",
        va="top",
        ha="right",
    )


def plot_cdf_panel(summary_df, max_features, xlabel, output_path):
    fig, ax = plt.subplots(figsize=(2, 2))
    bins = np.arange(0.5, max_features + 2.5, 1)
    counts, bin_edges = np.histogram(summary_df["mean_genes_to_threshold"], bins=bins)
    cdf = np.cumsum(counts) / counts.sum()
    ax.step(bin_edges[1:-1], cdf[:-1], where="mid")

    if np.any(cdf >= 0.5):
        idx_cross = np.argmax(cdf >= 0.5)
        x_cross = bin_edges[1:][idx_cross]
        ax.axvline(x_cross, color="red", linestyle=":", linewidth=1)

    ax.set_xlabel(xlabel, fontsize=7)
    ax.set_ylabel("CDF", fontsize=7)
    ax.set_ylim([0, 1])
    ax.tick_params(axis="both", labelsize=6)
    plt.tight_layout()
    save_png(fig, output_path)
    return fig, ax


In [ ]:
for path in CACHE_PATHS.values():
    require_path(path)

tmg_path = DATA_DIR / "TMG2"
TMG = TissueMultiGraph(basepath=str(tmg_path))

OptCellTx = TMG.get_tax("opt_cell")
OptRegionTx = TMG.get_tax("opt_region")

mi_c = np.load(CACHE_PATHS["mi_c"])
mi_r = np.load(CACHE_PATHS["mi_r"])
alphabet_results = load_pickle(CACHE_PATHS["alphabet_results"])
region_alphabet_results = load_pickle(CACHE_PATHS["region_alphabet_results"])
matrix_gene_cell = pd.read_csv(CACHE_PATHS["matrix_gene_cell"], index_col=0)
matrix_cell_region = pd.read_csv(CACHE_PATHS["matrix_cell_region"], index_col=0)

cell_feature_mat = OptCellTx.feature_mat.copy()[:, :500]
region_feature_mat = OptRegionTx.feature_mat.copy()
cmat = np.corrcoef(region_feature_mat.T)

rgb_region = np.array(rgb_array_to_hex(OptRegionTx.RGB))
rgb_cell = np.array(rgb_array_to_hex(OptCellTx.RGB))
rgb_gene = distinctipy.get_colors(cell_feature_mat.shape[1])

print(f"Loaded TMG from: {tmg_path}")
print(f"mi_c shape: {mi_c.shape}")
print(f"mi_r shape: {mi_r.shape}")
print(f"matrix_gene_cell shape: {matrix_gene_cell.shape}")
print(f"matrix_cell_region shape: {matrix_cell_region.shape}")


In [ ]:
Nvec_cell = [5000, 10000, 15000, 20000, 25000]
Cvec_cell = [1, 2, 3, 4, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 60, 70, 80, 90, 100, 125, 150, 200, 250, 300, 350, 400, 450, 500]

TMG.update_current_type(0, "opt_cell")
cell_entropy = list_entropy(TMG.Layers[0].Type)
mi_c_corrected = correct_mi_bias(
    mi_c,
    Nvec_cell,
    ([0.0, 0.0, 0.0], [cell_entropy, np.inf, 2.0]),
)

fig, ax = plt.subplots(figsize=(2, 2))
ax.errorbar(
    Cvec_cell,
    np.nanmean(mi_c_corrected, axis=1),
    yerr=np.nanstd(mi_c_corrected, axis=1) / np.sqrt(mi_c_corrected.shape[1]),
    fmt="o",
    markersize=3,
)
ax.set_xscale("log")
ax.axhline(y=cell_entropy, color="red", linestyle="--")
ax.set_xlabel("# of Genes", fontsize=7)
ax.set_ylabel("Mutual Information", fontsize=7)
ax.set_xlim([0.8, 1000])
ax.tick_params(axis="both", labelsize=6)
plt.tight_layout()
save_png(fig, PANEL_PATHS["a"])
plt.show()

PANEL_PATHS["a"]


In [ ]:
Nvec_region = [5000, 10000, 15000, 20000, 25000]
Cvec_region = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100, 110, 120, 130, 140, 150, 165]

TMG.update_current_type(0, "opt_region")
region_entropy = list_entropy(TMG.Layers[0].Type)
mi_r_corrected = correct_mi_bias(
    mi_r,
    Nvec_region,
    ([0.0, 0.0, 0.0], [region_entropy, np.inf, 2.0]),
)

fig, ax = plt.subplots(figsize=(2, 2))
ax.errorbar(
    Cvec_region,
    np.nanmean(mi_r_corrected, axis=1),
    yerr=np.nanstd(mi_r_corrected, axis=1) / np.sqrt(mi_r_corrected.shape[1]),
    fmt="o",
    markersize=3,
)
ax.set_xscale("log")
ax.axhline(y=region_entropy, color="red", linestyle="--")
ax.set_xlabel("# of Cell Types", fontsize=7)
ax.set_ylabel("Mutual Information", fontsize=7)
ax.set_xlim([0.8, 200])
ax.tick_params(axis="both", labelsize=6)
plt.tight_layout()
save_png(fig, PANEL_PATHS["b"])
plt.show()

PANEL_PATHS["b"]


In [ ]:
alphabet_summary = summarize_saturation_threshold(
    alphabet_results,
    threshold_bits=ALPHABET_THRESHOLD_BITS,
)
region_alphabet_summary = summarize_saturation_threshold(
    region_alphabet_results,
    threshold_bits=ALPHABET_THRESHOLD_BITS,
)

plot_cdf_panel(
    alphabet_summary,
    max_features=30,
    xlabel="# Genes per type",
    output_path=PANEL_PATHS["c"],
)
plt.show()

plot_cdf_panel(
    region_alphabet_summary,
    max_features=30,
    xlabel="# Cell Types per Region",
    output_path=PANEL_PATHS["d"],
)
plt.show()

PANEL_PATHS["c"], PANEL_PATHS["d"]


In [ ]:
fig, ax = plot_three_layers(
    region_cell_mat=matrix_cell_region.T,
    cell_gene_mat=matrix_gene_cell.T,
    cmat=cmat,
    rgb_region=rgb_region,
    rgb_cell=rgb_cell,
    rgb_gene=rgb_gene,
    tilt_angle_deg=66,
    depth_scale=0.35,
    layer_gap=0.4,
    gene_radius=0.01,
    node_radius=0.01,
    gene_weight_threshold=0.2,
    figsize=(3.6, 3.6),
    edge_alpha=0.1,
    cell_scale=1,
    hide_unconnected_genes=True,
    title="",
)
save_png(fig, PANEL_PATHS["e"])
plt.show()

PANEL_PATHS["e"]


In [ ]:
missing_panels = [label for label in "abcde" if not PANEL_PATHS[label].exists()]
if missing_panels:
    raise FileNotFoundError(f"Missing panel images: {missing_panels}")

fig = plt.figure(figsize=(10.5, 5.1), constrained_layout=True)
outer = fig.add_gridspec(
    nrows=2,
    ncols=3,
    width_ratios=[1.0, 1.0, 2.45],
    height_ratios=[1.0, 1.0],
)

ax_a = fig.add_subplot(outer[0, 0])
show_image(ax_a, PANEL_PATHS["a"])
add_panel_label(ax_a, "a")

ax_b = fig.add_subplot(outer[0, 1])
show_image(ax_b, PANEL_PATHS["b"])
add_panel_label(ax_b, "b")

ax_c = fig.add_subplot(outer[1, 0])
show_image(ax_c, PANEL_PATHS["c"])
add_panel_label(ax_c, "c")

ax_d = fig.add_subplot(outer[1, 1])
show_image(ax_d, PANEL_PATHS["d"])
add_panel_label(ax_d, "d")

ax_e = fig.add_subplot(outer[:, 2])
show_image(ax_e, PANEL_PATHS["e"])
add_panel_label(ax_e, "e")

save_png(fig, PANEL_PATHS["mock"])
plt.show()

PANEL_PATHS["mock"]
